# EX13 — NumPy Real-World Exercises

**What you'll learn in this notebook**
- Vectorized thinking: replacing loops with array operations (the #1 real-world NumPy skill)
- Broadcasting rules and why they matter for performance
- Boolean masking / fancy indexing to filter real datasets fast
- Common statistical & aggregation functions used in day-to-day data work
- A mini real-world case study: analyzing a week of store sales data

**How to use this notebook**
Each section has a short explanation, a worked example, then an exercise
marked `# TODO`. Try the TODO yourself before scrolling to the solution cell.

**Tip:** In real jobs, NumPy is rarely used alone for tabular data — it's the
engine *underneath* Pandas, images, and ML tensors. Getting comfortable with
vectorization here directly makes Pandas and PyTorch easier later.


## 1. Vectorization vs. Loops

Why: loops over Python lists are slow. NumPy pushes the loop into C.

**Pointer:** if you ever write `for i in range(len(arr)): arr[i] = ...` on a NumPy array, stop — there's almost always a vectorized way.

In [ ]:
import numpy as np

# Example: convert a week of daily temperatures (Celsius) to Fahrenheit
celsius = np.array([21.0, 23.5, 19.8, 25.1, 22.0, 20.3, 24.7])

# Slow (loop) way -- don't do this in practice
fahrenheit_loop = []
for c in celsius:
    fahrenheit_loop.append(c * 9/5 + 32)

# Fast (vectorized) way -- do this
fahrenheit_vectorized = celsius * 9/5 + 32
print(fahrenheit_vectorized)


### TODO 1
Given `prices` (USD) below, compute the price after a 12% discount, **without** a Python loop.

In [ ]:
prices = np.array([19.99, 5.49, 120.00, 3.75, 89.90])

# TODO: compute discounted_prices using vectorized ops
discounted_prices = None
print(discounted_prices)


<details><summary>Solution (click to expand)</summary>

```python
discounted_prices = prices * 0.88
```
</details>

## 2. Broadcasting

Broadcasting lets NumPy apply operations between arrays of different (but compatible) shapes without copying data.

**Pointer:** shapes are compatible right-to-left if each dimension is equal or one of them is 1.

In [ ]:
# Real-world use: normalize each column of a small dataset (feature scaling)
data = np.array([
    [10, 200, 1],
    [15, 220, 0],
    [ 9, 180, 1],
    [20, 260, 0],
])  # rows = samples, cols = features

col_mean = data.mean(axis=0)   # shape (3,)
col_std = data.std(axis=0)     # shape (3,)

normalized = (data - col_mean) / col_std   # broadcasting (4,3) with (3,)
print(normalized)


### TODO 2
A store tracks daily sales for 3 products across 5 days as a `(5, 3)` array.
Each product has a different profit margin (a `(3,)` array). Compute total daily profit per product using broadcasting (no loop).

In [ ]:
sales = np.array([
    [10, 5, 8],
    [12, 4, 9],
    [ 9, 6, 7],
    [15, 3, 10],
    [11, 7, 6],
])
margin = np.array([2.5, 4.0, 1.75])  # profit per unit sold, per product

# TODO: compute daily_profit, shape (5,3), then total_profit_per_product, shape (3,)
daily_profit = None
total_profit_per_product = None
print(total_profit_per_product)


<details><summary>Solution</summary>

```python
daily_profit = sales * margin
total_profit_per_product = daily_profit.sum(axis=0)
```
</details>

## 3. Boolean Masking & Fancy Indexing

This is how you filter real data fast — it's the NumPy equivalent of a SQL `WHERE` clause.

**Pointer:** `arr[mask]` returns a *copy*; be careful when you then try to modify it and expect the original to change.

In [ ]:
ages = np.array([22, 35, 58, 19, 46, 63, 27, 41])
# Example: everyone eligible for a senior discount (60+)
seniors = ages[ages >= 60]
print("Seniors:", seniors)

# Combine conditions with & and | (not `and`/`or`)
working_age = ages[(ages >= 18) & (ages < 65)]
print("Working age:", working_age)


### TODO 3
Given `transactions` (amounts, some negative = refunds), find:
1. All refund amounts
2. The count of transactions over $100
3. Replace all refunds with 0 in a *copy* of the array

In [ ]:
transactions = np.array([45.0, -12.5, 230.0, 99.99, -5.0, 150.0, 12.0])

# TODO
refunds = None
count_over_100 = None
cleaned = transactions.copy()
# TODO: set cleaned[negative positions] = 0
print(refunds, count_over_100, cleaned)


<details><summary>Solution</summary>

```python
refunds = transactions[transactions < 0]
count_over_100 = (transactions > 100).sum()
cleaned = transactions.copy()
cleaned[cleaned < 0] = 0
```
</details>

## 4. Aggregations You'll Use Constantly

`sum, mean, std, min, max, argmin, argmax, median, percentile` — know the `axis` parameter cold: `axis=0` collapses rows (down each column), `axis=1` collapses columns (across each row).

In [ ]:
matrix = np.array([[4, 2, 9], [1, 7, 3], [6, 5, 8]])
print("Column sums (axis=0):", matrix.sum(axis=0))
print("Row means (axis=1):", matrix.mean(axis=1))
print("Overall max:", matrix.max(), "at index", np.unravel_index(matrix.argmax(), matrix.shape))


## 5. Mini Case Study — A Week of Store Sales

Real-world scenario: you have units sold per product per day. Answer business questions using only vectorized NumPy.

In [ ]:
np.random.seed(0)
products = ["Coffee", "Tea", "Juice", "Water", "Soda"]
sales = np.random.randint(0, 50, size=(7, 5))  # 7 days x 5 products
print(sales)


### TODO 4 — Answer these using NumPy only
1. Which product sold the most **total** units for the week?
2. What was the best single day (highest total units across all products)?
3. Which days had **zero** sales for any product? (hint: `np.any`)
4. What's the day-over-day change in total units sold (hint: `np.diff`)?

In [ ]:
# TODO
best_product_idx = None
best_day_idx = None
zero_sales_days = None
day_over_day_change = None

print(products[best_product_idx] if best_product_idx is not None else None)


<details><summary>Solution</summary>

```python
totals_per_product = sales.sum(axis=0)
best_product_idx = totals_per_product.argmax()

totals_per_day = sales.sum(axis=1)
best_day_idx = totals_per_day.argmax()

zero_sales_days = np.where(np.any(sales == 0, axis=1))[0]

day_over_day_change = np.diff(totals_per_day)
```
</details>


## Key Takeaways
- Think in **whole-array operations**, not element-by-element loops.
- Broadcasting saves memory and code — learn the shape-compatibility rule.
- Boolean masks are your filtering tool; combine with `&`/`|`, not `and`/`or`.
- Always know which `axis` you're aggregating over.
- These same instincts transfer directly to Pandas (built on NumPy) and PyTorch tensors.
